# Aula 07 — Parte D: LSA e Busca Híbrida no Motor do Grupo (Baixada Santista)
**Projeto Integrador III — Motor de Busca Municipal e Histórico da Baixada Santista**

**Alunos:** Ivan Guimarães dos S. da Silva, Jorge Sugano Suzart e Gustavo Geminiano

**Grupo:** `ivan_gustavo_jorge` · *Kernel: R | Google Colaboratory (IRkernel) / GitHub do Grupo*


In [1]:
# 1. Setup oficial do motor07 e carregamento resiliente do corpus congelado
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor07.R")

REPO <- "https://raw.githubusercontent.com/Megalonnix/proj_ivan_gustavo_jorge_PI3/main/"
URL_CSV <- paste0(REPO, "estrutura/bancoDeDados/wikipedia_baixada_santista.csv")
arquivo_local <- "wikipedia_baixada_santista.csv"

if (!file.exists(arquivo_local)) {
  tryCatch({
    download.file(URL_CSV, destfile = arquivo_local, mode = "wb", quiet = TRUE)
  }, error = function(e) {
    cat("Aviso ao tentar download:", conditionMessage(e), "\n")
  })
}

corpus_raw <- tryCatch({
  read.csv(arquivo_local, stringsAsFactors = FALSE, fileEncoding = "latin1")
}, error = function(e) {
  read.csv(arquivo_local, stringsAsFactors = FALSE, encoding = "latin1")
})

source("config.R")

validos <- !is.na(corpus_raw$texto) & (nchar(corpus_raw$texto) > cfg$minimo)
cp_validos <- corpus_raw[validos, ]
docs   <- setNames(cp_validos$texto, cp_validos$id)
origem <- setNames(ifelse(grepl("guaruja", cp_validos$titulo, ignore.case = TRUE), "Guaruja",
                   ifelse(grepl("santos",  cp_validos$titulo, ignore.case = TRUE), "Santos", "Bertioga")),
                   cp_validos$id)

ix <- montar(docs, cfg)
cat("Corpus carregado com sucesso! Total de documentos validos:", length(docs), "\n")
cat("Matriz TF-IDF:", paste(dim(ix$w), collapse = " x "), "\n")
table(origem)
estado()


Corpus carregado com sucesso! Total de documentos validos: 132 
Matriz TF-IDF: 3493 x 132 
origem
Bertioga  Guaruja   Santos 
      17       29       86 

[1] "MOTOR_VERSAO: motor07 v1 2026-10-15"
| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `cfg$minimo` | numeric | 1 | 200 |
| `cfg$grupo` | character | 1 | ivan_gustavo_jorge |
| `cfg$limpar` | logical | 1 | TRUE |
| `cfg$acentos` | character | 1 | translit |
| `cfg$stopwords` | character | 27 | de, a, o, em, e, do, ... |
| `cfg$k1` | numeric | 1 | 1.2 |
| `cfg$b` | numeric | 1 | 0.75 |
| `cfg$juizes` | character | 1 | jorge |
| `cfg$limiar` | numeric | 1 | 2 |
| `cfg$prf_k` | numeric | 1 | 0 |
| `cfg$rocchio` | numeric, nomeado | 3 | a 1, b 0.75, g 0.15 |
| `cfg$modelo` | character | 1 | bm25 |
| `cfg$lsa_k` | numeric | 1 | 20 |
| `cfg$rrf_c` | numeric | 1 | 60 |
| `docs` | character, nomeado | 132 | 1 Guarujá é um município bras..., 2 Atualmente, a cidade de Gua..., ... |
| `ix$w` | matrix | 3493 x 132 | linhas: 

## Carregamento do Gabarito do Grupo e Consultas
Leitura do gabarito humano (`qrels_jorge_2026-10-13.csv`), necessidades de informação e separação em desenvolvimento (`dev`) e teste (`teste`).


In [1]:
# Leitura de qrels e necessidades
arqs <- c("qrels_jorge_2026-10-13.csv")
qrels <- ler_qrels(arqs, juiz = cfg$juizes)
nec   <- read.csv("necessidades.csv", stringsAsFactors = FALSE, fileEncoding = "UTF-8")
consultas <- setNames(nec$texto_consulta, nec$consulta)
R_q  <- sapply(qrels, function(g) sum(g >= cfg$limiar))
usar <- names(qrels)[R_q > 0]

dev   <- intersect(usar, nec$consulta[nec$conjunto == "dev"])
teste <- intersect(usar, nec$consulta[nec$conjunto == "teste"])

cat("Consultas ativas (R > 0):", paste(usar, collapse = ", "), "\n")
cat("Conjunto de Desenvolvimento:", paste(dev, collapse = ", "), "\n")
cat("Conjunto de Teste:", paste(teste, collapse = ", "), "\n")


Consultas ativas (R > 0): q01, q02, q03, q04, q05, q06, q07, q08, q09, q10, q11, q12 
Conjunto de Desenvolvimento: q01, q02, q03, q04, q05, q06, q07, q08 
Conjunto de Teste: q09, q10, q11, q12 


## Módulo 7 — O LSA no Corpus do Grupo: Uma Consulta, Lendo os Textos
Aplicação da SVD no corpus de 132 documentos, cálculo de energia retida e comparação detalhada do top-5 entre BM25 e LSA na consulta $q_{01}$ ("fundação de Santos").


In [1]:
# Funcoes do LSA da teoria
lsa <- function(ix, k) {
  s <- svd(ix$w)
  U <- s$u[, 1:k, drop = FALSE]; rownames(U) <- ix$vocab
  D <- s$v[, 1:k, drop = FALSE] %*% diag(s$d[1:k], nrow = k)
  rownames(D) <- colnames(ix$w)
  list(U = U, S = s$d[1:k], D = D)
}
ranking_lsa <- function(consulta, ix, cfg, modelo) {
  q0 <- vetor_consulta(preparar(consulta, cfg), ix$vocab, ix$idf_tfidf)
  qk <- t(modelo$U) %*% q0
  s  <- apply(modelo$D, 1, function(d) cosseno(qk, d))
  sort(s, decreasing = TRUE)
}

# Energia da SVD no corpus do grupo
s <- svd(ix$w)
for (k in c(10, 20, 50, 100, 132)) {
  en <- sum(s$d[1:k]^2) / sum(s$d^2)
  cat(sprintf("k = %3d: energia = %.4f (%.2f%%)\n", k, en, en*100))
}

# Comparacao na consulta q01
m20 <- lsa(ix, 20)
q <- "q01"
rl <- ranking_lsa(consultas[[q]], ix, cfg, m20)
rb <- ranking_bm25(consultas[[q]], ix, cfg)

cat("\nTop-5 lado a lado em 'fundacao de Santos':\n")
comp_top5 <- rbind(bm25 = names(rb)[1:5], lsa = names(rl)[1:5])
print(comp_top5)

so_lsa <- names(rl)[1:5][!names(rl)[1:5] %in% names(rb)[1:5]]
so_bm25 <- names(rb)[1:5][!names(rb)[1:5] %in% names(rl)[1:5]]
cat("Documentos que SO o LSA trouxe no top-5:", paste(so_lsa, collapse = ", "), "\n")
cat("Documentos que SO o BM25 trouxe no top-5:", paste(so_bm25, collapse = ", "), "\n")


k =  10: energia = 0.2183 (21.83%)
k =  20: energia = 0.3660 (36.60%)
k =  50: energia = 0.6598 (65.98%)
k = 100: energia = 0.9200 (92.00%)
k = 132: energia = 1.0000 (100.00%)

Top-5 lado a lado em 'fundacao de Santos':
     [,1] [,2] [,3]  [,4]  [,5]
bm25 "13" "55" "114" "87"  "83"
lsa  "55" "53" "31"  "121" "59"
Documentos que SO o LSA trouxe no top-5: 53, 31, 121, 59 
Documentos que SO o BM25 trouxe no top-5: 13, 114, 87, 83 


In [1]:
# Inspecao qualitativa dos textos exclusivos do LSA e do BM25
cat("=== EXCLUSIVOS DO LSA ===\n")
for (d in so_lsa) {
  palavras <- intersect(preparar(consultas[[q]], cfg), ix$tokens[[d]])
  cat(sprintf("Doc %s: grau=%s | palavras em comum (%d): %s\nTexto: %s...\n\n",
              d, as.character(qrels[[q]][d]), length(palavras), paste(palavras, collapse=", "),
              substr(docs[[d]], 1, 140)))
}

cat("=== EXCLUSIVOS DO BM25 ===\n")
for (d in so_bm25) {
  palavras <- intersect(preparar(consultas[[q]], cfg), ix$tokens[[d]])
  cat(sprintf("Doc %s: grau=%s | palavras em comum (%d): %s\nTexto: %s...\n\n",
              d, as.character(qrels[[q]][d]), length(palavras), paste(palavras, collapse=", "),
              substr(docs[[d]], 1, 140)))
}


=== EXCLUSIVOS DO LSA ===
Doc 53: grau=NA | palavras em comum (1): santos
Texto: No fim do século XVIII, a vila retoma o desenvolvimento e sua população começa a crescer. A construção da Calçada do Lorena - estrada de lig...

Doc 31: grau=NA | palavras em comum (1): santos
Texto: A outra parte, relevante e não-sazonal, advém do porto (margem esquerda do Porto de Santos) e atividades afins, tais como transporte. Devido ...

Doc 121: grau=NA | palavras em comum (0): 
Texto: Carvalho apoiou-se numa tradição lírica que nos remete a Victor Hugo e foi um grande divulgador da cidade como advogado e jurista,[137] e hoj...

Doc 59: grau=2 | palavras em comum (1): santos
Texto: Santos se tornou definitivamente uma cidade turística a partir dos anos 1910, com a construção dos hotéis Internacional e Parque Balneário e...

=== EXCLUSIVOS DO BM25 ===
Doc 13: grau=2 | palavras em comum (1): fundacao
Texto: O empreendimento é inaugurado em 2 de setembro de 1893 e torna-se reduto da classe alta paulist

## Módulo 8 — BM25, LSA e Híbrido em Todas as Consultas (Dev × Teste)
Avaliação do MAP no desenvolvimento ($q_{01}..q_{08}$) comparando BM25, LSA ($k=10, 20, 50$) e Híbrido via RRF ($c=60$). A decisão do modelo é tomada **exclusivamente no dev** e testada **uma única vez no teste**.


In [1]:
# Funcoes de fusao e avaliacao de modelos
rrf <- function(..., c = 60) {
  rks <- list(...)
  ids <- unique(unlist(lapply(rks, names)))
  s <- sapply(ids, function(d)
    sum(sapply(rks, function(r) {
      p <- which(names(r) == d)
      if (length(p)) 1 / (c + p) else 0 })))
  sort(s, decreasing = TRUE)
}

ranking_modelo <- function(txt, modelo, ix, cfg, m, c = 60) {
  rb <- ranking_bm25(txt, ix, cfg)
  if (modelo == "bm25") return(rb)
  rl <- ranking_lsa(txt, ix, cfg, m)
  if (modelo == "lsa") rl else rrf(rb, rl, c = c)
}

ap_modelo <- function(q, modelo, m, consultas, qrels, ix, cfg) {
  metricas(ranking_modelo(consultas[[q]], modelo, ix, cfg, m), qrels[[q]], cfg$limiar)[["AP"]]
}

ks <- c(10, 20, 50)
ms <- setNames(lapply(ks, function(k) lsa(ix, k)), paste0("k", ks))

# Tabela no Desenvolvimento (DEV)
tab_dev <- sapply(dev, function(q) c(
  bm25 = ap_modelo(q, "bm25", NULL, consultas, qrels, ix, cfg),
  lsa  = sapply(ms, function(m) ap_modelo(q, "lsa", m, consultas, qrels, ix, cfg)),
  hib  = sapply(ms, function(m) ap_modelo(q, "hibrido", m, consultas, qrels, ix, cfg))))

cat("=== AP POR CONSULTA NO DESENVOLVIMENTO ===\n")
print(round(tab_dev, 3))
cat("\n=== MAP NO DESENVOLVIMENTO (rowMeans) ===\n")
print(round(rowMeans(tab_dev), 3))


=== AP POR CONSULTA NO DESENVOLVIMENTO ===
          q01   q02   q03   q04   q05   q06   q07   q08
bm25    0.718 0.791 0.706 0.766 0.735 0.790 0.591 0.854
lsa.k10 0.289 0.209 0.196 0.227 0.471 0.556 0.177 0.466
lsa.k20 0.290 0.436 0.267 0.470 0.442 0.581 0.211 0.522
lsa.k50 0.395 0.593 0.469 0.444 0.610 0.569 0.339 0.627
hib.k10 0.581 0.562 0.421 0.536 0.680 0.753 0.386 0.763
hib.k20 0.561 0.730 0.556 0.730 0.687 0.747 0.423 0.783
hib.k50 0.600 0.748 0.621 0.707 0.722 0.746 0.524 0.804

=== MAP NO DESENVOLVIMENTO (rowMeans) ===
   bm25 lsa.k10 lsa.k20 lsa.k50 hib.k10 hib.k20 hib.k50 
  0.744   0.324   0.402   0.506   0.585   0.652   0.684 


In [1]:
# Execucao unica no conjunto de Teste (q09..q12)
escolha <- "bm25"   # Decisao tomada no dev: BM25 supera LSA e Hibrido em todas as consultas
k_esc   <- 20       # Dimensao representativa do LSA
m_esc   <- lsa(ix, k_esc)

tab_teste <- sapply(teste, function(q) c(
  bm25      = ap_modelo(q, "bm25", NULL, consultas, qrels, ix, cfg),
  escolhido = ap_modelo(q, escolha, m_esc, consultas, qrels, ix, cfg),
  hib_k20   = ap_modelo(q, "hibrido", m_esc, consultas, qrels, ix, cfg),
  hib_k50   = ap_modelo(q, "hibrido", ms[["k50"]], consultas, qrels, ix, cfg)))

cat("=== RESULTADOS NO CONJUNTO DE TESTE ===\n")
print(round(tab_teste, 3))
cat("\n=== MAP NO TESTE ===\n")
print(round(rowMeans(tab_teste), 3))


=== RESULTADOS NO CONJUNTO DE TESTE ===
            q09   q10   q11   q12
bm25      0.843 0.670 0.593 0.767
escolhido 0.843 0.670 0.593 0.767
hib_k20   0.813 0.655 0.477 0.621
hib_k50   0.867 0.641 0.494 0.749

=== MAP NO TESTE ===
     bm25 escolhido   hib_k20   hib_k50 
    0.718     0.718     0.642     0.688 


## Módulo 9 — A Decisão do Grupo, Registro em `config.R` e Homologação do Motor
**Decisão:** Manter `modelo = "bm25"` em produção, pois o BM25 supera o LSA e o Híbrido em todas as consultas dev (MAP $0{,}744$ vs $0{,}652$ e $0{,}402$) e no teste ($0{,}718$ vs $0{,}642$). Registram-se `lsa_k = 20` e `rrf_c = 60` no `config.R` para documentar a parametrização experimental avaliada.


In [1]:
# Homologacao e execucao da funcao oficial de ranking do motor
source("config.R")
cat("Parametros homologados em config.R:\n")
cat("cfg$modelo:", cfg$modelo, "| cfg$lsa_k:", cfg$lsa_k, "| cfg$rrf_c:", cfg$rrf_c, "\n\n")

ranking_motor <- function(txt, ix, cfg) {
  ranking_modelo(txt, cfg$modelo, ix, cfg, lsa(ix, cfg$lsa_k), c = cfg$rrf_c)
}

cat("Top-5 do motor em producao na consulta q01:\n")
round(head(ranking_motor(consultas[["q01"]], ix, cfg), 5), 4)


Parametros homologados em config.R:
cfg$modelo: bm25 | cfg$lsa_k: 20 | cfg$rrf_c: 60 

Top-5 do motor em producao na consulta q01:
    13     55    114     87     83 
5.1200 4.2680 0.8360 0.8260 0.8170 


In [1]:
# Anexacao automatica do estado final aos documentos
if (file.exists("00_FICHA_PROJETO.md")) anexar_estado("00_FICHA_PROJETO.md")
if (file.exists("aula07_parteD_consolidado.md")) anexar_estado("aula07_parteD_consolidado.md")
cat("Estados anexados com sucesso!\n")


Estados anexados com sucesso!

